In [1]:
%%capture
! wget -qnc https://github.com/ahlqui/VeloxChemColabs/raw/refs/heads/main/install.py
! python install.py
!pip install -q py3Dmol

In [2]:

import sys
python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
sys.path.append(f'/usr/local/lib/python{python_version}/site-packages/')

import veloxchem as vlx
from rdkit import Chem
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import math


# VeloxChem TS guesses for all FC mechanism steps

This corrected copy stores every result in `all_ts_results`, instead of displaying only one `ts_dict`.


In [ ]:

# Build ALL transition-state guesses and keep every result.e
# The old notebook only displayed ts_dict, which points to one result.
# Here each step is stored in all_ts_results[step_label].

# Optional but slower: uncomment this if you want VeloxChem to do a QM scan too.
# If you leave it commented, the summary will mostly report MM-scan information.
ts_guess = vlx.TransitionStateGuesser()
# ts_guess.do_qm_scan = True
# ts_guess.scf_scan = True  # use this instead only if your VeloxChem version uses scf_scan

# Common SMILES
paf = "CC(=O)NC(Cc1ccc(N)cc1)C(=O)NC"
enal = "CCC/C=C/C=O"

adduct_1_1 = "CC(=O)NC(Cc1ccc([NH2+][CH]([O-])/C=C/CCC)cc1)C(=O)NC"
carbinol_1_2 = "CC(=O)NC(Cc1ccc(N[CH](O)/C=C/CCC)cc1)C(=O)NC"
iminium_1_3 = "CC(=O)NC(Cc1ccc([NH+]=[CH]/C=C/CCC)cc1)C(=O)NC"

indole = "c1ccc2[nH]ccc2c1"
wheland_2_1 = "CC(=O)NC(Cc1ccc(N/C=C/C(CCC)C2C=[NH+]c3ccccc23)cc1)C(=O)NC"
enamine_2_2a = "CC(=O)NC(Cc1ccc(N/C=C/C(CCC)c2c[nH]c3ccccc23)cc1)C(=O)NC"
final_2_2b = "CC(=O)NC(Cc1ccc([NH+]=[CH]CC(CCC)c2c[nH]c3ccccc23)cc1)C(=O)NC"

water = "O"
hydronium = "[OH3+]"
hydroxide = "[OH-]"

# Each tuple is: label, reactant SMILES list, product SMILES list.
steps = [
    ("1.1",  [paf, enal],                         [adduct_1_1]),
    ("1.2",  [adduct_1_1, water, water],           [carbinol_1_2, hydronium, hydroxide]),
    ("1.3",  [carbinol_1_2, hydronium, hydroxide], [iminium_1_3, water, water, hydroxide]),
    ("2.1",  [iminium_1_3, indole],                [wheland_2_1]),
    ("2.2a", [wheland_2_1, hydroxide],             [enamine_2_2a, water]),
    ("2.2b", [enamine_2_2a, water],                [final_2_2b, hydroxide]),
]

def mols_from_smiles(smiles_list):
    return [vlx.Molecule.read_smiles(smi) for smi in smiles_list]

all_ts_results = {}
failed_ts_results = {}

for label, reactant_smiles, product_smiles in steps:
    print(f"\n================ Step {label} ================")
    try:
        reactants = mols_from_smiles(reactant_smiles)
        products = mols_from_smiles(product_smiles)
        result = ts_guess.find_transition_state(reactants, products)
        all_ts_results[label] = result
        globals()["ts_" + label.replace('.', '_')] = result
        print(f"Stored result as all_ts_results['{label}']")
    except Exception as exc:
        failed_ts_results[label] = exc
        print(f"FAILED step {label}: {type(exc).__name__}: {exc}")

print("\nFinished.")
print("Successful steps:", list(all_ts_results.keys()))
print("Failed steps:", {k: str(v) for k, v in failed_ts_results.items()})



================ Step 1.1 ================
* Info * Building forcefields. Disable mute_ff_build to see detailed output.                                              
                                                     Reaction summary                                                     
                                                    0 breaking bonds:                                                     
                                                                                                                          
                                                     1 forming bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                                          nv       ny    11 -     c       c3    40                                        
                                                                                               

In [4]:

# Summarize ALL available results.
# This does not depend on ts_guess.show_results(), so it works better in Colab.

SCALAR_TYPES = (int, float, str, bool)

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

# VeloxChem versions may use slightly different key names, so search flexibly.
def _find_first_key(dct, candidates):
    if not isinstance(dct, dict):
        return None, None
    lower_map = {str(k).lower(): k for k in dct.keys()}
    for cand in candidates:
        if cand.lower() in lower_map:
            k = lower_map[cand.lower()]
            return k, dct[k]
    # fallback: substring match
    for k, v in dct.items():
        kl = str(k).lower()
        if any(cand.lower() in kl for cand in candidates):
            return k, v
    return None, None

def _scan_items(result):
    scan = result.get("scan", {}) if isinstance(result, dict) else {}
    if not isinstance(scan, dict):
        return []
    items = []
    for lam, records in scan.items():
        if isinstance(records, dict):
            records = [records]
        if records is None:
            records = []
        for idx, rec in enumerate(records):
            if isinstance(rec, dict):
                items.append((lam, idx, rec))
    return items

def _best_from_scan(result, energy_names):
    best = None
    for lam, idx, rec in _scan_items(result):
        key, val = _find_first_key(rec, energy_names)
        e = _safe_float(val)
        if e is None:
            continue
        if best is None or e > best["energy"]:
            best = {"lambda": lam, "conformer": idx, "energy": e, "energy_key": key}
    return best

def summarize_result(label, result):
    keys = list(result.keys()) if isinstance(result, dict) else []
    xyz_key = "max_qm_xyz" if "max_qm_xyz" in keys else ("max_mm_xyz" if "max_mm_xyz" in keys else "")
    best_qm = _best_from_scan(result, ["qm_energy", "scf_energy", "qm e", "qm"])
    best_mm = _best_from_scan(result, ["mm_energy", "mm e", "openmm_energy", "energy"])

    scalar_bits = []
    for k in keys:
        v = result[k]
        if isinstance(v, SCALAR_TYPES) and any(w in str(k).lower() for w in ["lambda", "energy", "barrier"]):
            scalar_bits.append(f"{k}={v}")

    return {
        "step": label,
        "xyz_key": xyz_key,
        "scan_points": len(_scan_items(result)),
        "best_qm_lambda": "" if best_qm is None else best_qm["lambda"],
        "best_qm_energy": "" if best_qm is None else best_qm["energy"],
        "best_mm_lambda": "" if best_mm is None else best_mm["lambda"],
        "best_mm_energy": "" if best_mm is None else best_mm["energy"],
        "extra_scalars": "; ".join(scalar_bits),
        "keys": ", ".join(map(str, keys)),
    }

rows = [summarize_result(label, result) for label, result in all_ts_results.items()]

html = "<table><tr>" + "".join(f"<th>{h}</th>" for h in [
    "step", "xyz_key", "scan_points", "best_qm_lambda", "best_qm_energy", "best_mm_lambda", "best_mm_energy", "extra_scalars"
]) + "</tr>"
for r in rows:
    html += "<tr>" + "".join(f"<td>{r[h]}</td>" for h in [
        "step", "xyz_key", "scan_points", "best_qm_lambda", "best_qm_energy", "best_mm_lambda", "best_mm_energy", "extra_scalars"
    ]) + "</tr>"
html += "</table>"
display(HTML(html))

print("Available result keys by step:")
for r in rows:
    print(f"Step {r['step']}: {r['keys']}")


step,xyz_key,scan_points,best_qm_lambda,best_qm_energy,best_mm_lambda,best_mm_energy,extra_scalars
1.1,max_mm_xyz,21,,,,,max_mm_lambda=0.3
1.2,max_mm_xyz,21,,,,,max_mm_lambda=0.15
1.3,max_mm_xyz,21,,,,,max_mm_lambda=0.75
2.1,max_mm_xyz,21,,,,,max_mm_lambda=0.35
2.2a,max_mm_xyz,21,,,,,max_mm_lambda=0.0
2.2b,max_mm_xyz,21,,,,,max_mm_lambda=0.3


Available result keys by step:
Step 1.1: breaking_bonds, forming_bonds, static_bonds, reactant, product, lambda_vec, scan, max_mm_xyz, max_mm_lambda, min_mm_conformer_index
Step 1.2: breaking_bonds, forming_bonds, static_bonds, reactant, product, lambda_vec, scan, max_mm_xyz, max_mm_lambda, min_mm_conformer_index
Step 1.3: breaking_bonds, forming_bonds, static_bonds, reactant, product, lambda_vec, scan, max_mm_xyz, max_mm_lambda, min_mm_conformer_index
Step 2.1: breaking_bonds, forming_bonds, static_bonds, reactant, product, lambda_vec, scan, max_mm_xyz, max_mm_lambda, min_mm_conformer_index
Step 2.2a: breaking_bonds, forming_bonds, static_bonds, reactant, product, lambda_vec, scan, max_mm_xyz, max_mm_lambda, min_mm_conformer_index
Step 2.2b: breaking_bonds, forming_bonds, static_bonds, reactant, product, lambda_vec, scan, max_mm_xyz, max_mm_lambda, min_mm_conformer_index


In [5]:

# Try VeloxChem's built-in widget for EACH step.
# If it fails in Colab, use the custom py3Dmol viewers below.

for label, result in all_ts_results.items():
    print(f"\n================ VeloxChem show_results: Step {label} ================")
    try:
        ts_guess.show_results(result)
    except Exception as exc:
        print(f"show_results failed for step {label}: {type(exc).__name__}: {exc}")



================ VeloxChem show_results: Step 1.1 ================



================ VeloxChem show_results: Step 1.2 ================



================ VeloxChem show_results: Step 1.3 ================



================ VeloxChem show_results: Step 2.1 ================



================ VeloxChem show_results: Step 2.2a ================



================ VeloxChem show_results: Step 2.2b ================


In [ ]:

# Dropdown viewer for the best TS guess from every successful step.
# It uses max_qm_xyz when present, otherwise max_mm_xyz.

if not all_ts_results:
    print("No successful TS results found. Run the all_ts_results cell first.")
else:
    step_dropdown = widgets.Dropdown(
        options=list(all_ts_results.keys()),
        description="Step"
    )
    out_best = widgets.Output()

    def show_best(label):
        with out_best:
            clear_output(wait=True)
            result = all_ts_results[label]
            xyz_key = "max_qm_xyz" if "max_qm_xyz" in result else "max_mm_xyz"
            xyz = result.get(xyz_key, None)
            print(f"Step {label} | showing {xyz_key}")
            if xyz is None:
                print("No max_qm_xyz or max_mm_xyz found. Result keys:", list(result.keys()))
                return
            view = py3Dmol.view(width=650, height=500)
            view.addModel(xyz, "xyz")
            view.setStyle({"stick": {"radius": 0.12}, "sphere": {"scale": 0.25}})
            view.zoomTo()
            view.show()

    widgets.interact(show_best, label=step_dropdown)
    display(out_best)


interactive(children=(Dropdown(description='Step', options=('1.1', '1.2', '1.3', '2.1', '2.2a', '2.2b'), value…

Output()

In [7]:

# Dropdown + slider to inspect the full lambda scan for every step.
# This replaces the old viewer that only used ts_dict["scan"].

if not all_ts_results:
    print("No successful TS results found. Run the all_ts_results cell first.")
else:
    step_dropdown2 = widgets.Dropdown(
        options=list(all_ts_results.keys()),
        description="Step"
    )
    frame_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="frame")
    out_scan = widgets.Output()

    def get_scan_frames(label):
        result = all_ts_results[label]
        scan = result.get("scan", {})
        frames = []
        if isinstance(scan, dict):
            for lam in sorted(scan.keys(), key=lambda x: float(x)):
                records = scan[lam]
                if isinstance(records, dict):
                    records = [records]
                for idx, rec in enumerate(records):
                    if not isinstance(rec, dict):
                        continue
                    xyz = rec.get("xyz") or rec.get("mm_xyz") or rec.get("qm_xyz")
                    if xyz is not None:
                        frames.append((lam, idx, rec, xyz))
        return frames

    def reset_slider(*args):
        frames = get_scan_frames(step_dropdown2.value)
        frame_slider.max = max(len(frames) - 1, 0)
        frame_slider.value = 0

    def show_scan_frame(label, frame):
        with out_scan:
            clear_output(wait=True)
            frames = get_scan_frames(label)
            if not frames:
                print(f"No scan XYZ frames found for step {label}.")
                print("Result keys:", list(all_ts_results[label].keys()))
                return
            frame = min(frame, len(frames) - 1)
            lam, idx, rec, xyz = frames[frame]
            energy_items = {k: v for k, v in rec.items() if "energy" in str(k).lower() or str(k).lower() in ["qm", "mm"]}
            print(f"Step {label} | frame {frame+1}/{len(frames)} | lambda={float(lam):.2f} | conformer={idx}")
            if energy_items:
                print("Energies:", energy_items)
            view = py3Dmol.view(width=650, height=500)
            view.addModel(xyz, "xyz")
            view.setStyle({"stick": {"radius": 0.12}, "sphere": {"scale": 0.25}})
            view.zoomTo()
            view.show()

    step_dropdown2.observe(reset_slider, names="value")
    reset_slider()
    widgets.interact(show_scan_frame, label=step_dropdown2, frame=frame_slider)
    display(out_scan)


interactive(children=(Dropdown(description='Step', options=('1.1', '1.2', '1.3', '2.1', '2.2a', '2.2b'), value…

Output()